In [21]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
import unicodedata
import time
import pandas as pd

In [22]:
def set_up_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [23]:
def get_medicine_infomation(driver):
    
    info = {
        "hoat_chat": "",
        "quy_cach_dong_goi": "",
        "thuong_hieu": "",
        "xuat_xu": "",
        "thuoc_can_ke_toa": "",
        "dang_bao_che": "",
        "ham_luong": "",
        "nha_san_xuat": "",
        "tieu_chuan": "",
    }

    wait = WebDriverWait(driver, 0.5)
    for _ in range(4):
        if driver.find_elements(By.CSS_SELECTOR, "div.table-thongtinsanpham-border table.tbl-thongtinsanpham tbody tr"):
            break
        driver.execute_script("window.scrollBy(0, 350);")
        time.sleep(0.25)
    try:
        wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR, "div.table-thongtinsanpham-border table.tbl-thongtinsanpham tbody tr"
        )))

        rows = driver.find_elements(
            By.CSS_SELECTOR,
            "div.table-thongtinsanpham-border table.tbl-thongtinsanpham tbody tr"
        )

        raw = {}
        for row in rows:
            cells = row.find_elements(By.CSS_SELECTOR, "th, td")
            if len(cells) >= 2:
                k = cells[0].text.strip().rstrip(":")
                v = cells[1].text.strip()
                if k:
                    raw[k] = v

        # map label -> field cố định
        info["hoat_chat"] = raw.get("Hoạt chất", "")
        info["quy_cach_dong_goi"] = raw.get("Quy cách đóng gói", "")
        info["thuong_hieu"] = raw.get("Thương hiệu", "")
        info["xuat_xu"] = raw.get("Xuất xứ", "")
        info["thuoc_can_ke_toa"] = raw.get("Thuốc cần kê toa", "")
        info["dang_bao_che"] = raw.get("Dạng bào chế", "")
        info["ham_luong"] = raw.get("Hàm lượng", "")
        info["nha_san_xuat"] = raw.get("Nhà sản xuất", "")
        info["tieu_chuan"] = raw.get("Tiêu chuẩn", "")

    except Exception as e:
        print(f"Lỗi khi lấy bảng thông tin sản phẩm: {type(e).__name__}: {e}")

    return info


In [24]:
def expand_button(driver):
    try:
        wait = WebDriverWait(driver, 0.5)
        selector = "#main > div > div > div > div > div > div > div.col-12.col-md-8.sanpham_info.row > div.tab-sanpham.pt-4 > span"

       
        xem_them_btn = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, selector))
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center', inline: 'nearest'});",
            xem_them_btn
        )
        time.sleep(0.3)
        driver.execute_script("window.scrollBy(0, -120);")  
        time.sleep(0.2)

        
        wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
        try:
            xem_them_btn.click()
        except Exception:
            driver.execute_script("arguments[0].click();", xem_them_btn)

        time.sleep(0.5)

    except (NoSuchElementException, TimeoutException):
        print("Không tìm thấy nút 'Xem thêm'.")
        return None

In [25]:

from urllib.parse import urljoin

def get_image_links(driver):
    wait = WebDriverWait(driver, 0.25)
    image_urls = []
    seen = set()

    try:
        
        wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR, "div.sanpham_img, div.sanpham_img.overflow-hidden"
        )))

        
        img_elements = driver.find_elements(By.CSS_SELECTOR, """
            div.sanpham_img img,
            div.owl-carousel img,
            div.owl-stage-outer img,
            div.owl-item img
        """)

        for img in img_elements:
            
            src = (
                img.get_attribute("src")
                or img.get_attribute("data-src")
                or img.get_attribute("data-lazy")
                or img.get_attribute("data-original")
            )
            if not src:
                continue

            full_url = urljoin(driver.current_url, src.strip())
            
            if full_url.startswith("data:"):
                continue
            if any(x in full_url.lower() for x in ["logo", "icon", "zalo"]):
                continue

            if full_url not in seen:
                seen.add(full_url)
                image_urls.append(full_url)

       
        bg_elements = driver.find_elements(By.CSS_SELECTOR, "div.sanpham_img [style*='background-image']")
        for el in bg_elements:
            style = el.get_attribute("style") or ""
            if "url(" in style:
                raw = style.split("url(", 1)[1].split(")", 1)[0].strip("'\" ")
                if raw:
                    full_url = urljoin(driver.current_url, raw)
                    if full_url not in seen:
                        seen.add(full_url)
                        image_urls.append(full_url)

    except Exception as e:
        print(f"Lỗi khi lấy ảnh: {type(e).__name__}: {e}")

    return image_urls


In [26]:

def get_medicine_detail_info(driver):
    result = {
        "thanh_phan": "",
        "cong_dung_chi_dinh": "",
        "lieu_dung": "",
        "cach_dung": "",
        "luu_y": "",
        "chong_chi_dinh": "",
        "tac_dung_phu": "",
        "tuong_tac_thuoc": "",
        "Bao_quan": "",
    }

    wait = WebDriverWait(driver, 0.25)
    expand_button(driver)

    def norm_text(s: str) -> str:
        return unicodedata.normalize("NFC", (s or "")).strip().lower()

    try:
        root = wait.until(EC.visibility_of_element_located((
            By.CSS_SELECTOR, "#offcanvasbottom.show #data_detail_productid, #data_detail_productid"
        )))

        # Kéo thanh cuộn trong offcanvas xuống cuối để tải/hiện hết nội dung
        try:
            scroll_box = driver.find_element(
                By.CSS_SELECTOR,
                "#offcanvasbottom.show #load-canvas-bottom, #offcanvasbottom.show .offcanvas-body"
            )
        except NoSuchElementException:
            scroll_box = root

        last_top = -1
        for _ in range(20):
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollTop + 500;", scroll_box)
            time.sleep(0.2)
            cur_top = driver.execute_script("return arguments[0].scrollTop", scroll_box)
            max_top = driver.execute_script("return arguments[0].scrollHeight - arguments[0].clientHeight", scroll_box)
            if cur_top == last_top or cur_top >= max_top:
                break
            last_top = cur_top

        driver.execute_script("arguments[0].scrollTop = 0;", scroll_box)
        time.sleep(0.2)

        result["detail_raw_text"] = root.text.strip()

        nodes = root.find_elements(By.XPATH, ".//*[self::h2 or self::h3 or self::p or self::li]")
        current_title = None
        bucket = {}

        for el in nodes:
            tag = (el.tag_name or "").lower()
            txt = (el.get_attribute("textContent") or "").strip()
            if not txt:
                continue

            if tag in ("h2", "h3"):
                current_title = txt
                bucket.setdefault(current_title, [])
            else:
                if current_title is None:
                    current_title = "Mô tả"
                    bucket.setdefault(current_title, [])
                bucket[current_title].append(txt)

        for title, chunks in bucket.items():
            content = "\n".join(chunks).strip()
            t = norm_text(title)

            if "thành phần" in t:
                result["thanh_phan"] = content
            elif "liều dùng" in t:
                result["lieu_dung"] = content
            elif "cách dùng" in t:
                result["cach_dung"] = content
            elif "thận trọng" in t or "lưu ý" in t:
                result["luu_y"] = content
            elif "chống chỉ định" in t or ("không sử dụng" in t and "chỉ định" in t):
                result["chong_chi_dinh"] = content
            elif "công dụng" in t or "chỉ định" in t:
                result["cong_dung_chi_dinh"] = content
            elif "tác dụng phụ" in t:
                result["tac_dung_phu"] = content
            elif "tương tác" in t and "thuốc" in t:
                result["tuong_tac_thuoc"] = content
            elif "bảo quản" in t:
                result["Bao_quan"] = content

    except Exception as e:
        print(f"Lỗi khi lấy nội dung chi tiết: {type(e).__name__}: {e}")

    return result

In [27]:
def get_medicine_data(driver):
    wait = WebDriverWait(driver, 0.5)

    try: 
        name = driver.find_element(By.XPATH, "//*[@id='main']/div/div/div/div/div/div/div[1]/div[2]/div[1]").text.strip()
        time.sleep(0.5)
    except Exception:
        name = ""

    try:
        root = wait.until(EC.visibility_of_element_located((
            By.CSS_SELECTOR, "#offcanvasbottom.show #data_detail_productid, #data_detail_productid"
        )))

        raw_text = root.text.strip()

        # Parse theo từng section h2
        sections = {}
        current_key = None

        children = root.find_elements(By.XPATH, "./*")
        for el in children:
            tag = (el.tag_name or "").lower()
            txt = el.text.strip()
            if not txt:
                continue

            if tag == "h2":
                current_key = txt
                if current_key not in sections:
                    sections[current_key] = ""
            else:
                if current_key is None:
                    current_key = "Mô tả chung"
                    if current_key not in sections:
                        sections[current_key] = ""
                sections[current_key] = (sections[current_key] + "\n" + txt).strip()

        return {
            "name": name,
            "raw_text": raw_text,
            "sections": sections
        }

    except Exception as e:
        print(f"Lỗi khi lấy nội dung chi tiết: {type(e).__name__}: {e}")
        return {"name": name, "raw_text": "", "sections": {}}

In [28]:
# detail = get_medicine_detail_info(driver)
# print(detail["sections"].get("Thành phần", ""))
# print(detail["sections"].get("Công dụng (Chỉ định)", ""))

In [29]:

def crawl_medicine_data(
    input_csv="medicine_links_deduplicated.csv",
    output_csv="minhchau_medicines_data.csv",
    temp_csv="minhchau_medicines_data_temp.csv",
    save_every=10,
    sleep_sec=1.5
):
    df = pd.read_csv(input_csv)

    driver = set_up_driver()
    results = []

    try:
        total = len(df)

        for i, row in df.iterrows():
            category = str(row.get("category", "")).strip()
            url = str(row.get("link", "")).strip()

            if not url:
                continue

            try:
                driver.get(url)
                WebDriverWait(driver, 1).until(
                    EC.presence_of_element_located((By.TAG_NAME, "body"))
                )
                time.sleep(sleep_sec)

                try:
                    name = driver.find_element(By.XPATH, "//*[@id='main']/div/div/div/div/div/div/div[1]/div[2]/div[1]").text.strip()
                except Exception:
                    name = ""
                time.sleep(0.5)

                # Bảng thông tin
                info = get_medicine_infomation(driver) or {}

                # Chi tiết xem thêm
                detail = get_medicine_detail_info(driver) or {}

                # Ảnh
                images = get_image_links(driver) or []

                # Kết hợp 2 cột gốc (category, link) + thông tin crawl mới
                record = {
                    "category": category,
                    "link": url,
                    "name": name,
                    "images": " | ".join(images),
                    "error": ""
                }

                record.update(info)
                if isinstance(detail, dict):
                    record.update(detail)
                else:
                    record["detail_raw_text"] = str(detail)

                results.append(record)
                print(f"[{i+1}/{total}] crawled: {name if name else url}")

            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                print(f"[{i+1}/{total}] error: {err}")
                results.append({
                    "category": category,
                    "link": url,
                    "name": "",
                    "images": "",
                    "error": err
                })

            if (i + 1) % save_every == 0:
                pd.DataFrame(results).to_csv(temp_csv, index=False, encoding="utf-8-sig")
                print(f"Saved temp: {len(results)} records -> {temp_csv}")

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    final_df = pd.DataFrame(results)
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\nĐã lưu {len(final_df)} sản phẩm vào {output_csv}")
    return final_df

In [30]:
# Chạy hàm
crawled_data = crawl_medicine_data()

[1/9544] crawled: Apilevo 500 Apimed 3 vỉ x 10 viên
[2/9544] crawled: Vigentin 875/125 DT. Pharbaco 2 vỉ x 7 viên (Amoxicillin + Acid clavulanic)
[3/9544] crawled: Novofenti 200mg CPC1 Hà Nội 1 vỉ x 3 viên
[4/9544] crawled: Cotrimoxazol 480mg Thephaco 20 vỉ x 20 viên (Sulfamethoxazol + Trimethoprim)
[5/9544] crawled: Fungafin 10mg CPC1 Hà Nội lọ 15ml
[6/9544] crawled: Zurer-300 Davipharm 3 vỉ x 10 viên (Clindamycin)
[7/9544] crawled: Gravia Sup 200mg CPC1 Hà Nội 1 vỉ x 3 viên
[8/9544] crawled: Cefurovid 250 Vidipha 10 vỉ x 10 viên (Cefuroxim)
[9/9544] crawled: Cefuroxime Actavis 1.5g 5 lọ
[10/9544] crawled: Vagidequa 10mg CPC1 Hà Nội 1 vỉ x 6 viên
Saved temp: 10 records -> minhchau_medicines_data_temp.csv
[11/9544] crawled: Noverxar CPC1 Hà Nội 10 ống x 5ml
[12/9544] crawled: Micospray 20mg CPC1 Hà Nội tuýp 15ml
[13/9544] crawled: Skypodox-100 10 vỉ x 10 viên (Cefpodoxim)
[14/9544] crawled: Fentimeyer 1000 Meyer-BPC 1 vỉ x 10 viên
[15/9544] crawled: Mebifaclor 125mg/5ml Mebiphar 60ml (